# Model Training (qubit_TransmonCross_cap_matrix)
## Capacitance --> Quantum Metal

Inverse model with surrogate-defined loss for the Transmon Cross dataset.

Instead of training the inverse model to directly predict Quantum Metal parameters and comparing
against ground truth SQuADDS data (which penalizes valid alternative solutions), we chain the
inverse model with a frozen surrogate and compute loss in capacitance space.

**Pipeline:** Cap_input -> Inverse MLP -> Qiskit_params -> ScalerConversion -> Surrogate(frozen) -> Cap_reconstructed

**Loss:** MSE(Cap_input, Cap_reconstructed) + penalty for out-of-range predictions

The Transmon Cross dataset has all continuous parameters,
but the inverse model (ml_00/ml_01) and surrogate (ml_10/ml_11) use different scalers.
A ScalerConversionLayer handles the transform between scalers.

## Configuration

In [1]:
## the parameter file has the hyperparameters
## start there if you want to change the setup

from parameters_surrogate_defined_loss import *
print(KERAS_TUNER_TRIALS)

248


## Library

In [ ]:
import os, gc, joblib, json, time, sys, math, csv

## keep the tensorflow warnings down
## comment this out if you want the warnings back
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
from pandas import json_normalize
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model, Model
from tensorflow.keras.layers import Dense, Dropout, Input, LeakyReLU
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from tensorflow.python.client import device_lib

## imports that come up later
from tensorflow.keras import Sequential
from keras_tuner import BayesianOptimization
import keras_tuner as kt

seed = 0

## if the seed stays the same, the random numbers stay the same too
## you get the same random numbers every run
tf.random.set_seed(seed)
np.random.seed(seed)

## Check GPU

In [ ]:
## check what hardware tensorflow can see
print(device_lib.list_local_devices())

## run !{sys.executable} m pip install u pip
## run !{sys.executable} m pip install tensorflow[andcuda]
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)}')
for gpu in gpus:
    print(f'  {gpu}')

## check the cuda packages
!{sys.executable} -m pip list | egrep "tensorflow|nvidia-(cuda|cudnn|cublas|nccl)"

## Dataset

### Load

In [ ]:
## load the data saved from ml_00 notebook (inverse model format)
## for the transmon cross, ml_00 uses 'one_hot' encoding label but all params are continuous

## inputs are the capacitance values (what we want to invert)
X_train = np.load(f'{DATA_DIR}/npy/x_train_one_hot_encoding_augmented.npy', allow_pickle=True)
X_val   = np.load(f'{DATA_DIR}/npy/x_val_one_hot_encoding_augmented.npy', allow_pickle=True)
X_test  = np.load(f'{DATA_DIR}/npy/x_test_one_hot_encoding_augmented.npy', allow_pickle=True)

## qiskit metal params (all continuous for transmon cross)
y_train = np.load(f'{DATA_DIR}/npy/y_train_one_hot_encoding_augmented.npy', allow_pickle=True)
y_val   = np.load(f'{DATA_DIR}/npy/y_val_one_hot_encoding_augmented.npy', allow_pickle=True)
y_test  = np.load(f'{DATA_DIR}/npy/y_test_one_hot_encoding_augmented.npy', allow_pickle=True)

## column names
with open('metadata/X_names', 'r') as f:
    cap_column_names = f.read().splitlines()

qiskit_param_names = np.load('metadata/y_columns.npy', allow_pickle=True).astype(str).tolist()

print(f'Inputs (Capacitance):     {X_train.shape[1]} columns')
print(f'Outputs (Qiskit params):  {y_train.shape[1]} columns')
print(f'Training samples: {len(X_train)}')
print(f'\nCapacitance columns: {cap_column_names}')
print(f'Qiskit param columns: {qiskit_param_names}')

X_train = X_train.astype('float32')
y_train = y_train.astype('float32')
X_val   = X_val.astype('float32')
y_val   = y_val.astype('float32')
X_test  = X_test.astype('float32')
y_test  = y_test.astype('float32')

### Visualize

In [ ]:
print('X_train.shape: ', X_train.shape)
print('X_val.shape:   ', X_val.shape)
print('y_train.shape: ', y_train.shape)
print('y_val.shape:   ', y_val.shape)
print('y_train[0]: ', y_train[0])

display(X_train)
display(y_train)

## look at how it was split and decide if we like the split (we do for now)
total = len(X_train) + len(X_test) + len(X_val)
print(f'train: {len(X_train)} ({len(X_train)/total*100:.1f}%)')
print(f'val:   {len(X_val)} ({len(X_val)/total*100:.1f}%)')
print(f'test:  {len(X_test)} ({len(X_test)/total*100:.1f}%)')
print(f'total: {total}')

## bin the input data (capacitance values) and look at distribution

num_cols = X_train.shape[1]
num_rows = math.ceil(num_cols / 3)

fig, axes = plt.subplots(num_rows, 3, figsize=(10, 3 * num_rows))
axes = axes.ravel()
for i in range(num_cols):
    axes[i].hist(X_train[:, i], bins=30, edgecolor='black', alpha=0.7)
    axes[i].set_title(cap_column_names[i] if i < len(cap_column_names) else f'col_{i}')
for j in range(num_cols, len(axes)):
    fig.delaxes(axes[j])
plt.suptitle('Input distribution (capacitance, scaled)')
plt.tight_layout()
plt.show()

steps_per_epoch = int(np.ceil(len(X_train) / TRAIN_BATCH_SIZE))
LR_DECAY_STEPS = steps_per_epoch * 20   ## decay every ~20 epochs
print(f'Steps per epoch: {steps_per_epoch}, LR decay steps: {LR_DECAY_STEPS}')

## MLP

### Create model

Reccomended to download a third party app like "Sleep control Center" or "Amphetamine" to prevent computer from sleeping during training

### Make scaler conversion layer

The inverse model (ml_00/ml_01) and surrogate model (ml_10/ml_11) may use different MinMaxScalers
because they were fit on different data splits or processing pipelines.
This layer does the affine transform: `linear_scaled = oh_scaled * scale_a + scale_b`
to convert between the two scaler spaces. All operations are differentiable.

In [ ]:
## determine column mapping between inverse model output and surrogate input
## the inverse model outputs qiskit params in ml_00 scaler space
## the surrogate expects them in ml_10 scaler space

n_qiskit_params = y_train.shape[1]
print(f'Number of Quantum Metal params: {n_qiskit_params}')
print(f'Param names: {qiskit_param_names}')

## compute affine conversion constants between ml_00 and ml_10 scaler spaces
## for each param linear_scaled = oh_scaled * scale_a + scale_b
## where scale_a = oh_range / lin_range, scale_b = (oh_min lin_min) / lin_range

scale_a = np.ones(n_qiskit_params, dtype=np.float32)
scale_b = np.zeros(n_qiskit_params, dtype=np.float32)

for i, col in enumerate(qiskit_param_names):
    ## try loading both scalers
    oh_path = f'scalers/scaler_y_{col}_one_hot_encoding.save'
    lin_path = f'scalers/scaler_y_linear_{col}.save'
    
    if os.path.exists(oh_path) and os.path.exists(lin_path):
        oh_sc  = joblib.load(oh_path)
        lin_sc = joblib.load(lin_path)
        oh_min, oh_range   = float(oh_sc.data_min_[0]), float(oh_sc.data_range_[0])
        lin_min, lin_range = float(lin_sc.data_min_[0]), float(lin_sc.data_range_[0])
        scale_a[i] = oh_range / lin_range
        scale_b[i] = (oh_min - lin_min) / lin_range
        print(f'  {col}: a={scale_a[i]:.6f}, b={scale_b[i]:.6f}')
    else:
        print(f'  {col}: using identity (missing scaler file)')

if np.allclose(scale_a, 1.0) and np.allclose(scale_b, 0.0):
    print('\nScalers are identical - conversion is identity (no transform needed)')
else:
    print('\nScalers differ - affine conversion will be applied')

## load and freeze the surrogate model so its not updated during training
SURROGATE_PATH = 'model/best_keras_model_model2_surrogate.keras'
surrogate = load_model(SURROGATE_PATH, compile=False)

## freeze it
surrogate.trainable = False
for layer in surrogate.layers:
    layer.trainable = False

surr_input_dim = surrogate.input_shape[-1]
print(f'\nSurrogate loaded from {SURROGATE_PATH}')
print(f'Surrogate expects {surr_input_dim} inputs, inverse model outputs {n_qiskit_params}')
assert surr_input_dim == n_qiskit_params, f'Dimension mismatch!'
surrogate.summary()

## build the scaler conversion layer
## converts inverse model output (ml_00 scaler space) to surrogate input (ml_10 scaler space)
## all operations are differentiable so gradients flow through

class ScalerConversionLayer(tf.keras.layers.Layer):
    def __init__(self, scale_a, scale_b, **kwargs):
        kwargs.setdefault('trainable', False)
        super().__init__(**kwargs)
        self._scale_a = tf.constant(scale_a, dtype=tf.float32)
        self._scale_b = tf.constant(scale_b, dtype=tf.float32)
        self._cfg = dict(
            scale_a=list(scale_a) if hasattr(scale_a, '__iter__') else scale_a,
            scale_b=list(scale_b) if hasattr(scale_b, '__iter__') else scale_b)

    def call(self, inputs):
        a = tf.cast(self._scale_a, inputs.dtype)
        b = tf.cast(self._scale_b, inputs.dtype)
        return inputs * a + b

    def get_config(self):
        config = super().get_config()
        config.update(self._cfg)
        return config

scaler_converter = ScalerConversionLayer(
    scale_a=scale_a, scale_b=scale_b, name='scaler_conversion')

## quick sanity test
test_input = tf.constant(y_train[:3])
test_output = scaler_converter(test_input)
print(f'Conversion test: input shape {test_input.shape} -> output shape {test_output.shape}')

## penalizes the inverse model for predicting qiskit metal values outside
## the training data range [0, 1] (since data is MinMaxScaled).
## this prevents the model from finding "adversarial" inputs to the surrogate
## that look good in capacitance space but don't correspond to real designs.

PENALTY_WEIGHT = 0.1  ## tune this increase if params are still wild, decrease if reconstruction suffers

def qiskit_range_penalty(y_true_dummy, y_pred):
    """Penalize predictions outside [0, 1] range.
    y_true_dummy is ignored (we pass zeros as dummy targets).
    y_pred is the inverse model's raw output in scaled space."""
    below = tf.nn.relu(-y_pred)
    above = tf.nn.relu(y_pred - 1.0)
    return tf.reduce_mean(below ** 2 + above ** 2)

## prebuild dummy targets for training
dummy_y_train = np.zeros_like(y_train)
dummy_y_val   = np.zeros_like(y_val)
dummy_y_test  = np.zeros_like(y_test)

print(f'Penalty weight: {PENALTY_WEIGHT}')
print(f'Dummy targets shape: {dummy_y_train.shape}')

### Create Model by Hand

In [ ]:
## just checkin to make sure everything looks good still, we want float32
print(X_train.dtype, X_train.shape)
print(y_train.dtype, y_train.shape)

if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    ## model shape string for file naming
    model_shape = f'surrogate_loss_{X_train.shape[1]}in_{y_train.shape[1]}out'
    print(model_shape)

if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    ## build inverse model that takes capacitance > qiskit metal params
    inverse_model = Sequential(name='inverse_model')
    inverse_model.add(Input(shape=(X_train.shape[1],), name='cap_input'))
    
    for i, n in enumerate(NEURONS_PER_LAYER):
        inverse_model.add(Dense(n, name='fc{}'.format(i),
                        kernel_initializer='lecun_uniform',
                        kernel_regularizer=tf.keras.regularizers.l2(1e-4)))
        inverse_model.add(LeakyReLU(negative_slope=0.01, name='leaky_relu{}'.format(i)))
        inverse_model.add(Dropout(rate=TRAIN_DROPOUT_RATE, name='dropout{}'.format(i)))
    
    ## output layer predicts qiskit metal params
    inverse_model.add(Dense(y_train.shape[1], activation='linear', name='qiskit_output',
                    kernel_initializer='lecun_uniform'))
    
    ## twooutput combined model
    ## output 1 reconstructed capacitance (through frozen surrogate) main loss
    ## output 2 raw qiskit params range penalty loss
    combined_input = Input(shape=(X_train.shape[1],), name='combined_input')
    predicted_qiskit = inverse_model(combined_input)
    predicted_qiskit_converted = scaler_converter(predicted_qiskit)  ## affine conversion
    reconstructed_cap = surrogate(predicted_qiskit_converted)
    model = Model(
        inputs=combined_input,
        outputs=[reconstructed_cap, predicted_qiskit],
        name='combined_model'
    )
    
    print('Inverse model (outputs Quantum Metal params):')
    inverse_model.summary()
    print('\nCombined model (two outputs: cap reconstruction + Qiskit params for penalty):')
    model.summary()

if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=LR_INITIAL,  
        decay_steps=LR_DECAY_STEPS,        
        decay_rate=LR_DECAY_RATE,          
        staircase=LR_STAIRCASE             
    )
    
    ## two losses reconstruction + range penalty on qiskit params
    model.compile(
        optimizer=tf.optimizers.Adam(learning_rate=lr_schedule),  
        loss=[TRAIN_LOSS, qiskit_range_penalty],
        loss_weights=[1.0, PENALTY_WEIGHT],
        metrics={model.output_names[0]: [TRAIN_LOSS]}
    )

if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    !mkdir -p model
    best_model_file = 'model/{}_best_model.keras'.format(model_shape)
    last_model_file = 'model/{}_last_model.keras'.format(model_shape)
    print('Best model will be saved to:', best_model_file)

Enable training (`train_and_save`) to overwrite the model file.

In [24]:
train_and_save = True

We use Adam optimizer, minimize the loss specified in parameters, and early stop.

#### Training

In [ ]:
if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    class TrainingPlot(tf.keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.losses = []
            self.val_losses = []
        def on_epoch_end(self, epoch, logs=None):
            self.losses.append(logs.get('loss'))
            self.val_losses.append(logs.get('val_loss'))

    class LearningRateMonitor(tf.keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.learning_rates = []
        def on_epoch_end(self, epoch, logs=None):
            lr = self.model.optimizer.learning_rate
            if callable(lr):
                lr = lr(self.model.optimizer.iterations)
            self.learning_rates.append(float(tf.keras.backend.get_value(lr)))

%%time

## train the model two targets
## 1. cap reconstruction (input capacitance is the target)
## 2. dummy zeros for the penalty loss (penalty only uses predictions, ignores target)
history = None  
if not KERAS_TUNER and not SWEEP_PARAM_NUM and not SWEEP_DATA_AMOUNT:
    if train_and_save: 
        early_stopping = EarlyStopping(
            monitor='val_loss',
            mode='min',
            patience=TRAIN_EARLY_STOPPING_PATIENCE,
            verbose=1
        )
    
        plot_callback = TrainingPlot()
        lr_monitor = LearningRateMonitor()
        
        model_checkpoint = ModelCheckpoint(
            filepath=best_model_file,          
            monitor='val_loss',
            mode='min',
            save_best_only=True,
            verbose=0
        )

        history = model.fit(
            np.asarray(X_train),
            [np.asarray(X_train), dummy_y_train],
            epochs=400,                   
            batch_size=TRAIN_BATCH_SIZE,  
            validation_data=(np.asarray(X_val), [np.asarray(X_val), dummy_y_val]),  
            callbacks=[early_stopping, model_checkpoint, plot_callback, lr_monitor],  
            verbose=1
        )
        
        model.save(last_model_file)

Load the saved best model and use it from now on.

In [27]:
if not KERAS_TUNER and not SWEEP_PARAM_NUM and not SWEEP_DATA_AMOUNT:
    model = load_model(best_model_file, custom_objects={
        'ScalerConversionLayer': ScalerConversionLayer,
        'qiskit_range_penalty': qiskit_range_penalty})

### Sweep total number of parameters to find the right range

In [28]:
if SWEEP_PARAM_NUM:
    qiskit_dim = y_train.shape[1]
    cap_dim = X_train.shape[1]
    
    def build_combined_mlp(neurons_per_layer):
        inv = Sequential(name='inverse_model')
        inv.add(Input(shape=(cap_dim,), name='cap_input'))
        for i, n in enumerate(neurons_per_layer):
            inv.add(Dense(n, name=f'fc{i}', kernel_initializer='lecun_uniform',
                            kernel_regularizer=tf.keras.regularizers.l2(1e-4)))
            inv.add(LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}'))
            inv.add(Dropout(rate=TRAIN_DROPOUT_RATE, name=f'dropout{i}'))
        inv.add(Dense(qiskit_dim, activation='linear', name='qiskit_output',
                        kernel_initializer='lecun_uniform'))
        combined_input = Input(shape=(cap_dim,))
        qiskit_out = inv(combined_input)
        qiskit_conv = scaler_converter(qiskit_out)
        cap_recon = surrogate(qiskit_conv)
        return Model(combined_input, [cap_recon, qiskit_out])
    
    def make_optimizer():
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=LR_INITIAL,
            decay_steps=LR_DECAY_STEPS,
            decay_rate=LR_DECAY_RATE,
            staircase=LR_STAIRCASE)
        return tf.optimizers.Adam(learning_rate=lr_schedule)
    
    def train_one_config(neurons_per_layer, seed=0):
        tf.keras.backend.clear_session()
        tf.random.set_seed(seed)
        np.random.seed(seed)
        global surrogate, scaler_converter
        surrogate = load_model(SURROGATE_PATH, compile=False)
        surrogate.trainable = False
        for layer in surrogate.layers:
            layer.trainable = False
        scaler_converter = ScalerConversionLayer(
            scale_a=scale_a, scale_b=scale_b, name='scaler_conversion')
        combined = build_combined_mlp(neurons_per_layer)
        combined.compile(optimizer=make_optimizer(),
                        loss=[TRAIN_LOSS, qiskit_range_penalty],
                        loss_weights=[1.0, PENALTY_WEIGHT])
        early_stopping = EarlyStopping(
            monitor='val_loss', mode='min',
            patience=TRAIN_EARLY_STOPPING_PATIENCE, verbose=0,
            restore_best_weights=True)
        history = combined.fit(
            np.asarray(X_train), [np.asarray(X_train), dummy_y_train],
            validation_data=(np.asarray(X_val), [np.asarray(X_val), dummy_y_val]),
            epochs=400, batch_size=TRAIN_BATCH_SIZE,
            callbacks=[early_stopping], verbose=0)
        best_val = min(history.history['val_loss'])
        return combined, history, best_val

### Sweep amount of data used in training

In [29]:
if SWEEP_DATA_AMOUNT:
    FIXED_DEPTH = 1
    FIXED_WIDTH = 1024
    FIXED_NEURONS = [FIXED_WIDTH] * FIXED_DEPTH
    print(f'Sweep data amount with fixed architecture: {FIXED_NEURONS}')

### Keras Tuner to Find Best Hyperparameters

Run this if you want to use keras tuner to make the model rather than doing it by hand

In [ ]:

if KERAS_TUNER and not SWEEP_PARAM_NUM:
    cap_dim = X_train.shape[1]
    qiskit_dim = y_train.shape[1]
    
    def build_hypermodel(hp):
        tf.keras.backend.clear_session()
        gc.collect()
        
        ## reload surrogate + rebuild conversion layer after clear_session
        surr = load_model(SURROGATE_PATH, compile=False)
        surr.trainable = False
        for layer in surr.layers:
            layer.trainable = False
        converter = ScalerConversionLayer(
            scale_a=scale_a, scale_b=scale_b, name='scaler_conversion')
        
        n_layers = hp.Int('n_layers', min_value=1, max_value=4, default=2)
        neurons_per_layer = [hp.Int(f'neurons_{i}', min_value=64, max_value=1024, step=64) for i in range(n_layers)]
        dropout_rate = hp.Float('dropout_rate', 0.0, 0.3, step=0.05)
        l2_reg = hp.Float('l2_reg', 1e-6, 1e-2, sampling='LOG', default=1e-4)
        lr_initial = hp.Float('learning_rate', 1e-3, 1e-1, sampling='LOG', default=1e-2)
        use_batchnorm = hp.Boolean('use_batchnorm', default=True)
        penalty_wt = hp.Float('penalty_weight', 0.01, 1.0, sampling='LOG', default=0.1)
        
        ## build the inverse model
        inv = Sequential(name='inverse_model')
        inv.add(Input(shape=(cap_dim,), name='cap_input'))
        for i, n_units in enumerate(neurons_per_layer):
            inv.add(Dense(n_units, name=f'fc{i}', kernel_initializer='he_normal',
                            kernel_regularizer=tf.keras.regularizers.l2(l2_reg)))
            if use_batchnorm:
                inv.add(tf.keras.layers.BatchNormalization(name=f'bn{i}'))
            inv.add(LeakyReLU(negative_slope=0.01, name=f'leaky_relu{i}'))
            inv.add(Dropout(rate=dropout_rate, name=f'dropout{i}'))
        inv.add(Dense(qiskit_dim, name='qiskit_output', kernel_initializer='he_normal'))
        
        ## twooutput model
        combined_input = Input(shape=(cap_dim,), name='combined_input')
        qiskit_out = inv(combined_input)
        qiskit_conv = converter(qiskit_out)
        cap_recon = surr(qiskit_conv)
        combined = Model(combined_input, [cap_recon, qiskit_out], name='combined_model')
        combined.compile(
            optimizer=tf.optimizers.Adam(learning_rate=lr_initial),
            loss=[TRAIN_LOSS, qiskit_range_penalty],
            loss_weights=[1.0, penalty_wt],
        )
        return combined

if KERAS_TUNER and not SWEEP_PARAM_NUM:
    tuner = BayesianOptimization(
        build_hypermodel,
        objective='val_loss',
        max_trials=KERAS_TUNER_TRIALS,
        seed=seed,
        directory='kt_dir',
        project_name='transmon_cross_surrogate_loss'
    )

if KERAS_TUNER and not SWEEP_PARAM_NUM:
    early_stopping = EarlyStopping(
        monitor='val_loss', mode='min',
        patience=TRAIN_EARLY_STOPPING_PATIENCE, verbose=1)
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1)

if KERAS_TUNER and not SWEEP_PARAM_NUM:
    tuner.search(
        np.asarray(X_train), [np.asarray(X_train), dummy_y_train],
        epochs=400, batch_size=TRAIN_BATCH_SIZE,
        validation_data=(np.asarray(X_val), [np.asarray(X_val), dummy_y_val]),
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )

encoding = 'surrogate_defined_loss'
if KERAS_TUNER and not SWEEP_PARAM_NUM:
    os.makedirs('model', exist_ok=True)
    best_model_file = f'model/best_keras_model_{encoding}.keras'
    best_combined = tuner.get_best_models(1)[0]
    best_combined.save(best_model_file)
    
    inverse_only = best_combined.get_layer('inverse_model')
    inverse_only.save(f'model/best_inverse_model_{encoding}.keras')
    print(f'Saved combined model: {best_model_file}')
    print(f'Saved inverse-only model: model/best_inverse_model_{encoding}.keras')
    
    tf.keras.backend.clear_session()
    gc.collect()
    with tf.device('/CPU:0'):
        loaded_model = load_model(best_model_file, compile=False,
                                  custom_objects={'ScalerConversionLayer': ScalerConversionLayer})

### View the model

In [36]:
if KERAS_TUNER and not SWEEP_PARAM_NUM:
    print('Combined model (inverse + conversion + frozen surrogate):')
    best_combined.summary()

if not KERAS_TUNER and not SWEEP_PARAM_NUM:
    print('Combined model:')
    model.summary()

Combined model (inverse + conversion + frozen surrogate):


Model: "combined_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ combined_input (InputLayer)     │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ inverse_model (Sequential)      │ (None, 3)              │           643 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ scaler_conversion               │ (None, 3)              │             0 │
│ (ScalerConversionLayer)         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 6)              │         3,526 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,169 (16.29 KB)

 Trainable params: 643 (2.51 KB)

 Non-trainable params: 3,526 (13.77 KB)

### Evaluation

Plot training history.

### Visualize gradients for best model

In [ ]:
if KERAS_TUNER and not SWEEP_PARAM_NUM and VISUALIZE_GRADIENTS:
    class GradientNormLogger(tf.keras.callbacks.Callback):
        def __init__(self, x_probe, y_probe, layer_name_prefixes=('fc0',), log_every=1, verbose=0):
            super().__init__()
            self.x_probe = x_probe
            self.y_probe = y_probe
            self.prefixes = layer_name_prefixes
            self.log_every = log_every
            self.verbose = verbose
            self.records = []
        def on_epoch_end(self, epoch, logs=None):
            if epoch % self.log_every != 0: return
            with tf.GradientTape() as tape:
                preds = self.model(self.x_probe, training=True)
                loss = self.model.compute_loss(self.x_probe, self.y_probe, preds)
            grads = tape.gradient(loss, self.model.trainable_variables)
            rec = {'epoch': epoch, 'loss': float(loss)}
            for var, g in zip(self.model.trainable_variables, grads):
                if g is None: continue
                for pfx in self.prefixes:
                    if pfx in var.name:
                        rec[f'grad_norm_{var.name}'] = float(tf.norm(g))
            self.records.append(rec)
        def to_csv(self, path):
            pd.DataFrame(self.records).to_csv(path, index=False)

if KERAS_TUNER and not SWEEP_PARAM_NUM and VISUALIZE_GRADIENTS:
    probe_n = min(256, len(X_train))
    x_probe = np.asarray(X_train[:probe_n])
    y_probe = [np.asarray(X_train[:probe_n]), np.zeros((probe_n, y_train.shape[1]), dtype='float32')]
    grad_logger = GradientNormLogger(x_probe=x_probe, y_probe=y_probe,
        layer_name_prefixes=('fc0', 'qiskit_output'), log_every=1, verbose=1)
    best_hp = tuner.get_best_hyperparameters(1)[0]
    model = tuner.hypermodel.build(best_hp)
    lr_monitor = LearningRateMonitor()
    history = model.fit(
        np.asarray(X_train), [np.asarray(X_train), dummy_y_train],
        epochs=400, batch_size=TRAIN_BATCH_SIZE,
        validation_data=(np.asarray(X_val), [np.asarray(X_val), dummy_y_val]),
        callbacks=[early_stopping, lr_monitor, grad_logger], verbose=1)
    grad_logger.to_csv(f'plots/{encoding}_gradients.csv')
    del model; tf.keras.backend.clear_session(); gc.collect()

if KERAS_TUNER and not SWEEP_PARAM_NUM and VISUALIZE_GRADIENTS:
    dfg = pd.DataFrame(grad_logger.records)
    plt.figure(figsize=(12, 4))
    for col in dfg.columns:
        if col.startswith('grad_norm_'):
            plt.plot(dfg['epoch'], dfg[col], label=col.replace('grad_norm_', ''))
    plt.xlabel('Epoch'); plt.ylabel('Gradient Norm'); plt.legend(fontsize=6)
    plt.title('Gradient norms during training'); plt.tight_layout()
    plt.savefig(f'plots/{encoding}_gradient_norms.pdf'); plt.show()

### Look at best model

In [ ]:
if KERAS_TUNER and not SWEEP_PARAM_NUM and not VISUALIZE_GRADIENTS:
    class LearningRateMonitor(tf.keras.callbacks.Callback):
        def on_train_begin(self, logs=None):
            self.learning_rates = []
        def on_epoch_end(self, epoch, logs=None):
            lr = self.model.optimizer.learning_rate
            if callable(lr):
                lr = lr(self.model.optimizer.iterations)
            self.learning_rates.append(float(tf.keras.backend.get_value(lr)))

    best_hp = tuner.get_best_hyperparameters(1)[0]
    model = tuner.hypermodel.build(best_hp)
    lr_monitor = LearningRateMonitor()
    history = model.fit(
        np.asarray(X_train), [np.asarray(X_train), dummy_y_train],
        epochs=400, batch_size=TRAIN_BATCH_SIZE,
        validation_data=(np.asarray(X_val), [np.asarray(X_val), dummy_y_val]),
        callbacks=[early_stopping, lr_monitor], verbose=1)
    del model; tf.keras.backend.clear_session(); gc.collect()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Surrogate-defined loss (total: reconstruction + range penalty)')
plt.ylabel('Loss'); plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='best')
plt.tight_layout(); plt.savefig(f'plots/surrogate_loss_history.pdf'); plt.show()

plt.plot(lr_monitor.learning_rates)
plt.title('Learning Rate over Epochs')
plt.xlabel('Epoch'); plt.ylabel('Learning Rate')
plt.tight_layout(); plt.savefig(f'plots/surrogate_loss_learning_rate.pdf'); plt.show()

### Test set evaluation

In [42]:
## evaluate on test set
tf.keras.backend.clear_session()
gc.collect()

def get_loss(eval_out):
    if isinstance(eval_out, dict):
        return float(eval_out.get('loss', list(eval_out.values())[0]))
    if isinstance(eval_out, (list, tuple, np.ndarray)):
        return float(eval_out[0])
    return float(eval_out)

combined_model = load_model(best_model_file, compile=False,
    custom_objects={'ScalerConversionLayer': ScalerConversionLayer})
combined_model.compile(optimizer='adam',
    loss=[TRAIN_LOSS, qiskit_range_penalty],
    loss_weights=[1.0, PENALTY_WEIGHT])
eval_result = combined_model.evaluate(
    np.asarray(X_test), [np.asarray(X_test), dummy_y_test])
total_loss = float(eval_result[0])
cap_recon_loss = float(eval_result[1])
penalty_loss = float(eval_result[2])

print(f'Total test loss: {total_loss}')
print(f'  Reconstruction loss ({TRAIN_LOSS}): {cap_recon_loss}')
print(f'  Range penalty loss: {penalty_loss} (weighted: {penalty_loss * PENALTY_WEIGHT})')

## check outofrange
inverse_model = combined_model.get_layer('inverse_model')
qiskit_pred = inverse_model.predict(np.asarray(X_test), verbose=0)
out_of_range = np.sum((qiskit_pred < 0) | (qiskit_pred > 1))
total_values = qiskit_pred.size
print(f'Out-of-range values: {out_of_range}/{total_values} ({100*out_of_range/total_values:.1f}%)')

test_loss_result = cap_recon_loss

10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step - inverse_model_loss: 1.4824e-06 - loss: 0.0043 - sequential_loss: 0.0043
Total test loss: 0.004306318238377571
  Reconstruction loss (mae): 0.004306368064135313
  Range penalty loss: 1.4824441905147978e-06 (weighted: 1.482444190514798e-07)
Out-of-range values: 15/873 (1.7%)


## Compare predictions vs. test set

In [ ]:
csv_data = [[
    DATA_AUGMENTATION,
    'surrogate_loss_model',
    'InverseModel_SurrogateLoss',
    test_loss_result,
    TRAIN_LOSS,
    TRAIN_BATCH_SIZE,
]]
with open('results/training/test_results.csv', 'a', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(csv_data)
    print(f'Appended to results/training/test_results.csv: {csv_data}')

## predictaroo on test set
tf.keras.backend.clear_session()
gc.collect()

with tf.device('/CPU:0'):
    combined_model = load_model(best_model_file, compile=False,
        custom_objects={'ScalerConversionLayer': ScalerConversionLayer})
    predictions = combined_model.predict(np.asarray(X_test), verbose=0)
    if isinstance(predictions, list):
        cap_reconstructed = predictions[0]
        qiskit_predicted = predictions[1]
    else:
        cap_reconstructed = predictions
        inverse_model = combined_model.get_layer('inverse_model')
        qiskit_predicted = inverse_model.predict(np.asarray(X_test), verbose=0)

## look at how well the reconstructed capacitance matches the input capacitance

X_test_cur = np.asarray(X_test)
y_test_cur = np.asarray(y_test)  ## ground truth qiskit params (for reference)
cap_recon  = np.asarray(cap_reconstructed)
qiskit_pred = np.asarray(qiskit_predicted)

n_samples, n_cap_cols = X_test_cur.shape
n_qiskit_cols = qiskit_pred.shape[1]
n_samples_to_show = 3

## reconstruction errors (capacitance space this is what we trained on)
cap_abs_errors = np.abs(X_test_cur - cap_recon)

print('capacitance reconstruction for the loss')
for i in range(n_samples_to_show):
    rows = []
    for j in range(n_cap_cols):
        label = cap_column_names[j] if j < len(cap_column_names) else f'cap_col_{j}'
        rows.append({'param': label, 'ref': X_test_cur[i,j],
                     'pred': cap_recon[i,j], 'abs_error': cap_abs_errors[i,j]})
    print(f'- Sample {i} - Capacitance reconstruction (scaled)')
    print(pd.DataFrame(rows).to_string(index=False))
    
    ## predicted qiskit params
    print(f'\n  Predicted Quantum Metal params (scaled):')
    for j, col_name in enumerate(qiskit_param_names):
        short = col_name.replace('design_options.', '')
        print(f'    {short:40s}  pred={qiskit_pred[i,j]:.6f}  ref={y_test_cur[i,j]:.6f}  err={abs(qiskit_pred[i,j]-y_test_cur[i,j]):.6f}')
    print()

### Unscaled test vs predictions

In [46]:
## unscale everything and look at errors in real units that we can actually make sense of
with open('metadata/X_names', 'r') as f:
    cap_names = f.read().splitlines()
qiskit_names = np.load('metadata/y_columns.npy', allow_pickle=True).astype(str).tolist()

## unscale input capacitance
x_scaler_prefix = 'scaler_X'
X_test_unscaled = np.asarray(X_test_cur.copy())
for i in range(X_test_unscaled.shape[0]):
    for j in range(X_test_unscaled.shape[1]):
        cap_name = cap_names[j] if j < len(cap_names) else f'col_{j}'
        scaler = joblib.load(f'scalers/{x_scaler_prefix}_{cap_name}.save')
        X_test_unscaled[i, j] = scaler.inverse_transform([[X_test_unscaled[i, j]]])[0][0]

## unscale reconstructed capacitance
cap_recon_unscaled = np.asarray(cap_recon.copy())
for i in range(cap_recon_unscaled.shape[0]):
    for j in range(cap_recon_unscaled.shape[1]):
        cap_name = cap_names[j] if j < len(cap_names) else f'col_{j}'
        scaler = joblib.load(f'scalers/{x_scaler_prefix}_{cap_name}.save')
        cap_recon_unscaled[i, j] = scaler.inverse_transform([[cap_recon_unscaled[i, j]]])[0][0]

## unscale qiskit param predictions (using ml_00 one_hot scalers)
qiskit_pred_unscaled = np.asarray(qiskit_pred.copy())
y_test_unscaled = np.asarray(y_test_cur.copy())
for i in range(qiskit_pred_unscaled.shape[0]):
    for j in range(qiskit_pred_unscaled.shape[1]):
        col_name = qiskit_names[j] if j < len(qiskit_names) else f'col_{j}'
        scaler = joblib.load(f'scalers/scaler_y_{col_name}_one_hot_encoding.save')
        qiskit_pred_unscaled[i, j] = scaler.inverse_transform([[qiskit_pred_unscaled[i, j]]])[0][0]
        y_test_unscaled[i, j] = scaler.inverse_transform([[y_test_unscaled[i, j]]])[0][0]

n_samples_to_show = 3
cap_abs_unscaled = np.abs(X_test_unscaled - cap_recon_unscaled)

print('unscaled capacitance reconstruction')
for i in range(n_samples_to_show):
    rows = []
    for j in range(X_test_unscaled.shape[1]):
        rows.append({'param': cap_names[j], 'ref_unscaled': X_test_unscaled[i,j],
                     'pred_unscaled': cap_recon_unscaled[i,j], 'abs_error': cap_abs_unscaled[i,j]})
    print(f'- Sample {i} (Unscaled) - Capacitance reconstruction')
    print(pd.DataFrame(rows).to_string(index=False))
    
    print(f'\n  Predicted Quantum Metal params (unscaled):')
    for j, col_name in enumerate(qiskit_names):
        short = col_name.replace('design_options.', '')
        print(f'    {short:40s}  pred={qiskit_pred_unscaled[i,j]:.6f}  ref={y_test_unscaled[i,j]:.6f}  err={abs(qiskit_pred_unscaled[i,j]-y_test_unscaled[i,j]):.6f}')
    
    print(f'\n  Reference Quantum Metal params (unscaled):')
    for j, col_name in enumerate(qiskit_names):
        short = col_name.replace('design_options.', '')
        print(f'    {short:40s}  ref={y_test_unscaled[i,j]:.6f}')
    print()

print('Unscaled cap reconstruction error stats:')
print('  min:', float(cap_abs_unscaled.min()),
      ' median:', float(np.median(cap_abs_unscaled)),
      ' max:', float(cap_abs_unscaled.max()))

########## Unscaled Capacitance Reconstruction #############
— Sample 0 (Unscaled) — Capacitance reconstruction
           param  ref_unscaled  pred_unscaled  abs_error
 cross_to_ground    133.517151     133.581512   0.064362
  claw_to_ground     96.702980      96.794426   0.091446
   cross_to_claw      5.987050       5.967897   0.019153
  cross_to_cross    133.517151     133.554718   0.037567
    claw_to_claw    103.122643     103.214592   0.091949
ground_to_ground    287.806000     286.044098   1.761902

  Predicted Quantum Metal params (unscaled):
    connection_pads.readout.claw_length       pred=0.000180  ref=0.000180  err=0.000000
    connection_pads.readout.ground_spacing    pred=0.000005  ref=0.000005  err=0.000000
    cross_length                              pred=0.000261  ref=0.000260  err=0.000001

  Reference Quantum Metal params (unscaled):
    connection_pads.readout.claw_length       ref=0.000180
    connection_pads.readout.ground_spacing    ref=0.000005
    cross_lengt